# BMCS2003/BMCS2203/BMCS2074 Artificial Intelligence
## Predicting Sepsis Development Within a 6-Hour Window
#### Group Members: Chang Han Yean (SVM), Elwin Goh Yao Zu (KNN), Kaizen Soh (Decision Tree)

# 1.0 Import Necessary Libraries

In [1]:
import os
import joblib
import pandas as pd  # For handling and processing datasets
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For creating visual plots
import seaborn as sns  # For statistical data visualization
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC  # Faster linear SVM for large datasets
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

MODEL_DIR = "DataTraining/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("Libraries imported successfully!")

Libraries imported successfully!


# 2.0 Data Preprocessing

## 2.1 Dataset Preparation (Load Dataset)

In [2]:
# Load dataset
df = pd.read_csv("Sepsis_dataset.csv")
print(f"Dataset shape: {df.shape}")

Dataset shape: (546123, 44)


### 2.2 Inspect Dataset


In [3]:
# Display dataset with first and last 5 rows
display(df)

,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,68.54,0.0,NaN,NaN,-0.02,1.0,0.0,17072.0
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,...,NaN,NaN,68.54,0.0,NaN,NaN,-0.02,2.0,0.0,17072.0
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,...,NaN,NaN,68.54,0.0,NaN,NaN,-0.02,3.0,0.0,17072.0
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,...,NaN,NaN,68.54,0.0,NaN,NaN,-0.02,4.0,0.0,17072.0
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,...,NaN,330.0,68.54,0.0,NaN,NaN,-0.02,5.0,0.0,17072.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546118,23,23,141.0,95.0,NaN,NaN,79.0,NaN,23.0,NaN,...,NaN,NaN,70.93,1.0,1.0,0.0,-313.53,26.0,0.0,2314.0
546119,24,24,94.0,98.0,NaN,NaN,89.0,NaN,21.0,NaN,...,NaN,NaN,70.93,1.0,1.0,0.0,-313.53,27.0,0.0,2314.0
546120,25,25,104.0,94.0,35.67,NaN,78.0,NaN,24.0,NaN,...,NaN,NaN,70.93,1.0,1.0,0.0,-313.53,28.0,0.0,2314.0
546121,26,26,104.0,94.0,NaN,NaN,NaN,NaN,22.0,NaN,...,NaN,NaN,70.93,1.0,1.0,0.0,-313.53,29.0,0.0,2314.0




## 2.2 Data Cleaning

In [4]:
df.describe()


,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
count,546123.000000,546123.000000,504000.000000,480319.000000,184511.000000,462920.000000,490527.000000,284671.000000,492675.000000,0.0,...,4164.000000,35591.000000,546122.000000,546122.000000,274708.000000,274708.000000,546122.000000,546122.000000,546122.000000,546122.000000
mean,25.287530,25.287530,84.936506,97.258570,37.028197,121.097229,78.865428,60.102131,18.749110,NaN,...,294.148823,198.769724,63.008405,0.577182,0.507477,0.492523,-52.003997,27.159966,0.021698,10158.505592
std,27.764845,27.764845,16.933876,2.935284,0.783216,21.506672,15.057816,12.611608,5.379793,NaN,...,161.515125,108.848196,16.150164,0.494007,0.499945,0.499945,148.651563,28.070839,0.145697,5897.221150
min,0.000000,0.000000,20.000000,20.000000,21.000000,22.000000,20.000000,20.000000,1.000000,NaN,...,34.000000,5.000000,18.110000,0.000000,0.000000,0.000000,-3710.660000,1.000000,0.000000,1.000000
25%,9.000000,9.000000,73.000000,96.000000,36.560000,105.000000,68.330000,51.500000,15.000000,NaN,...,184.000000,126.000000,52.670000,0.000000,0.000000,0.000000,-38.010000,11.000000,0.000000,5024.000000
50%,20.000000,20.000000,84.000000,98.000000,37.060000,119.000000,77.000000,59.000000,18.000000,NaN,...,248.000000,181.000000,65.270000,1.000000,1.000000,0.000000,-2.590000,21.000000,0.000000,10131.000000
75%,33.000000,33.000000,96.000000,99.500000,37.560000,135.000000,87.670000,67.000000,22.000000,NaN,...,360.000000,246.000000,76.020000,1.000000,1.000000,1.000000,-0.020000,35.000000,0.000000,15246.000000
max,335.000000,335.000000,223.000000,100.000000,42.220000,274.000000,300.000000,298.000000,69.000000,NaN,...,1760.000000,1783.000000,89.000000,1.000000,1.000000,1.000000,23.990000,336.000000,1.000000,20643.000000


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 546123 entries, 0 to 546122
Data columns (total 44 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        546123 non-null  int64  
 1   Hour              546123 non-null  int64  
 2   HR                504000 non-null  float64
 3   O2Sat             480319 non-null  float64
 4   Temp              184511 non-null  float64
 5   SBP               462920 non-null  float64
 6   MAP               490527 non-null  float64
 7   DBP               284671 non-null  float64
 8   Resp              492675 non-null  float64
 9   EtCO2             0 non-null       float64
 10  BaseExcess        56478 non-null   float64
 11  HCO3              43875 non-null   float64
 12  FiO2              76575 non-null   float64
 13  pH                62185 non-null   float64
 14  PaCO2             47562 non-null   float64
 15  SaO2              26680 non-null   float64
 16  AST               8179 non-null

### 2.2.1 Check any missing value

In [6]:
#check each column how many null values has
df.isnull().sum()

Unnamed: 0               0
Hour                     0
HR                   42123
O2Sat                65804
Temp                361612
SBP                  83203
MAP                  55596
DBP                 261452
Resp                 53448
EtCO2               546123
BaseExcess          489645
HCO3                502248
FiO2                469548
pH                  483938
PaCO2               498561
SaO2                519443
AST                 537944
BUN                 501673
Alkalinephos        538173
Calcium             518937
Chloride            500815
Creatinine          509911
Bilirubin_direct    545305
Glucose             479815
Lactate             527440
Magnesium           503732
Phosphate           518517
Potassium           487114
Bilirubin_total     539405
TroponinI           545435
Hct                 481844
Hgb                 498055
PTT                 519684
WBC                 505194
Fibrinogen          541959
Platelets           510532
Age                      1
G

In [ ]:
# rows, column
df.shape

(546123, 44)

In [8]:
df.columns

Index(['Unnamed: 0', 'Hour', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP',
       'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2',
       'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine',
       'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate',
       'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC',
       'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2',
       'HospAdmTime', 'ICULOS', 'SepsisLabel', 'Patient_ID'],
      dtype='str')

In [9]:
#check the +/- result value
df['SepsisLabel'].value_counts()

SepsisLabel
0.0    534272
1.0     11850
Name: count, dtype: int64

### 2.2.2 Remove duplicate rows

In [10]:
# Count duplicate rows
print("Duplicate rows before removal:", df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

# Confirm removal
print("Duplicate rows after removal:", df.duplicated().sum())
print("Records remaining after cleaning:", df.shape[0])

Duplicate rows before removal: 0
Duplicate rows after removal: 0
Records remaining after cleaning: 546123


## 2.3 Handling Missing Values

In [12]:
# Identify target column
target_col = "SepsisLabel"

# 1. Drop ID columns and columns that are completely empty (e.g. EtCO2)
df = df.drop(columns=["Patient_ID", "Unnamed: 0"], errors="ignore")
df = df.dropna(axis=1, how="all")

# 2. Drop rows with missing target
rows_before = len(df)
df = df.dropna(subset=[target_col])
print(f"Rows removed (missing target): {rows_before - len(df)}")

# 3. Fill remaining missing feature values with medians of the column
feature_medians = df.drop(columns=[target_col]).median(numeric_only=True)
df_clean = df.copy()
df_clean[feature_medians.index] = df_clean[feature_medians.index].fillna(feature_medians)

# 4. Verify no missing values remain
missing_total = df_clean.isnull().sum().sum()
print(f"Remaining missing values: {missing_total} (should be 0)")
print(f"Clean dataset shape: {df_clean.shape}")
print("\nClass distribution:")
print(df_clean[target_col].value_counts(normalize=True).rename("proportion"))

Rows removed (missing target): 1
Remaining missing values: 0 (should be 0)
Clean dataset shape: (546122, 41)

Class distribution:
SepsisLabel
0.0    0.978302
1.0    0.021698
Name: proportion, dtype: float64


In [13]:
# Check every column missing value
summary_table = pd.DataFrame({
    'Column Name': df_clean.columns,
    'Data Type': df_clean.dtypes.values,
    'Missing Values': df_clean.isnull().sum().values,
    'Non-Null Count': df_clean.notnull().sum().values
})

display(summary_table)

,Column Name,Data Type,Missing Values,Non-Null Count
0,Hour,int64,0,546122
1,HR,float64,0,546122
2,O2Sat,float64,0,546122
3,Temp,float64,0,546122
4,SBP,float64,0,546122
5,MAP,float64,0,546122
6,DBP,float64,0,546122
7,Resp,float64,0,546122
8,BaseExcess,float64,0,546122
9,HCO3,float64,0,546122


## 3.0 Data Splitting

Split the cleaned data into training and testing sets before scaling or SMOTE to avoid data leakage.

In [14]:
# Define features (X) and target (y)
X = df_clean.drop(columns=[target_col]) # all input feature
y = df_clean[target_col].astype(int) # target value SepsisLabel

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Feature matrix shape: (546122, 40)
Target distribution:
SepsisLabel
0    534272
1     11850
Name: count, dtype: int64


In [15]:
# 80/20 stratified split keeps the sepsis class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")
print(f"\nTraining class distribution:\n{y_train.value_counts()}")
print(f"\nTesting class distribution:\n{y_test.value_counts()}")

Training set: 436897 samples
Testing set:  109225 samples

Training class distribution:
SepsisLabel
0    427417
1      9480
Name: count, dtype: int64

Testing class distribution:
SepsisLabel
0    106855
1      2370
Name: count, dtype: int64


In [ ]:
# Save the 20% test set for the Streamlit app (Try sample patient tab)
TEST_SAMPLES_PATH = os.path.join("DataTraining", "test_samples.csv")
os.makedirs("DataTraining", exist_ok=True)

df_test = X_test.copy()
df_test[target_col] = y_test.values
df_test.to_csv(TEST_SAMPLES_PATH, index=False)

print(f"Saved {len(df_test):,} test samples to {TEST_SAMPLES_PATH}")
print(df_test[target_col].value_counts())


### 3.1 Feature Scaling

KNN and SVM are distance-based models, so features must be scaled to the same range.
We fit the scaler on the **training set only**, then apply it to the test set.

In [16]:
# put every column into similiar scale
#Treat all column to a similiar range so model compare them fairly
#if no do scale knn and svm will look at the higest number and ignore the lowest number
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling complete.")
print(f"Scaled training set shape: {X_train_scaled.shape}")
print(f"Scaled testing set shape:  {X_test_scaled.shape}")
print(f"Sample scaled means (should be ~0): {X_train_scaled.mean(axis=0)[:5]}")

Feature scaling complete.
Scaled training set shape: (436897, 40)
Scaled testing set shape:  (109225, 40)
Sample scaled means (should be ~0): [-2.78429278e-17 -1.11394480e-15  5.98753055e-16  6.96398463e-15
 -7.71535336e-16]


### 3.2 Handle Class Imbalance (SMOTE)

SMOTE is creating fake sample data for the minority class.
Applied in training data only


It take one real data of the minority class and finds the nearest neighbors data. SMOTE pick a random spot on line between these two data and create. repeat until it's balanced

In [17]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("SMOTE resampling complete.")
print(f"Training set before SMOTE: {X_train_scaled.shape[0]} samples")
print(f"Training set after SMOTE:  {X_train_resampled.shape[0]} samples")
print("\nClass distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

SMOTE resampling complete.
Training set before SMOTE: 436897 samples
Training set after SMOTE:  854834 samples

Class distribution after SMOTE:
SepsisLabel
0    427417
1    427417
Name: count, dtype: int64


### 3.3 Preprocessing Summary

The data is now ready for model training. Use these variables in the KNN, SVM, and Decision Tree sections:

| Variable | Purpose |
|---|---|
| `X_train_resampled`, `y_train_resampled` | Balanced, scaled training data (all 3 models) |
| `X_test_scaled`, `y_test` | Scaled test data for evaluation (all 3 models) |

In [18]:
preprocessing_summary = pd.DataFrame(
    {
        "Stage": [
            "After cleaning",
            "Training set",
            "Test set",
            "After SMOTE (train only)",
        ],
        "Rows": [
            df_clean.shape[0],
            X_train.shape[0],
            X_test.shape[0],
            X_train_resampled.shape[0],
        ],
        "Features": [
            X.shape[1],
            X_train.shape[1],
            X_test.shape[1],
            X_train_resampled.shape[1],
        ],
    }
)

display(preprocessing_summary)
print("\nPreprocessing complete. Ready for model training.")

,Stage,Rows,Features
0,After cleaning,546122,40
1,Training set,436897,40
2,Test set,109225,40
3,After SMOTE (train only),854834,40



Preprocessing complete. Ready for model training.


In [19]:
# Preview first 5 rows of the cleaned feature matrix
display(df_clean.head())

,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,BaseExcess,HCO3,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,0,84.0,98.0,37.06,119.0,77.0,59.0,18.0,0.0,24.0,...,10.8,248.0,181.0,68.54,0.0,1.0,0.0,-0.02,1.0,0.0
1,1,65.0,100.0,37.06,119.0,72.0,59.0,16.5,0.0,24.0,...,10.8,248.0,181.0,68.54,0.0,1.0,0.0,-0.02,2.0,0.0
2,2,78.0,100.0,37.06,119.0,42.5,59.0,18.0,0.0,24.0,...,10.8,248.0,181.0,68.54,0.0,1.0,0.0,-0.02,3.0,0.0
3,3,73.0,100.0,37.06,119.0,77.0,59.0,17.0,0.0,24.0,...,10.8,248.0,181.0,68.54,0.0,1.0,0.0,-0.02,4.0,0.0
4,4,70.0,100.0,37.06,129.0,74.0,69.0,14.0,0.0,26.0,...,11.3,248.0,330.0,68.54,0.0,1.0,0.0,-0.02,5.0,0.0


# 4.0 Model Classification and Training

Same approach as senior project: **Pipeline (scaler + model)** so each saved `.pkl` includes scaling.

- **Train:** `X_train_smote`, `y_train_smote` (SMOTE-balanced, **unscaled**)
- **Test:** `X_test`, `y_test` (real test data, **unscaled** — pipeline scales automatically)
- **Save (4.5):** `svm_model.pkl`, `knn_model.pkl`, `dt_model.pkl` only — no separate `scaler.pkl`

In [20]:
# Shared setup for all models
stratified_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# SMOTE on unscaled training data — scaler lives inside each Pipeline (like senior project)
X_train_smote, y_train_smote = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(f"Pipeline training data: {X_train_smote.shape[0]} rows, {X_train_smote.shape[1]} features")


def model_metrics(y_true, y_pred):
    """Return accuracy, precision, recall, F1, and false negative rate (FNR)."""
    cm = confusion_matrix(y_true, y_pred)
    fnr = cm[1][0] / (cm[1][0] + cm[1][1]) if (cm[1][0] + cm[1][1]) > 0 else 0.0
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "FNR": fnr,
    }


print("Model evaluation helper and CV splitter ready.")

Model evaluation helper and CV splitter ready.


### 4.1 Support Vector Machine (SVM)

In [21]:
# 4.1 SVM — Chang Han Yean
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(random_state=42, max_iter=3000)),
])

param_grid_svm = {
    "svm__C": [0.1, 1, 10],
}

grid_svm = GridSearchCV(
    svm_pipeline,
    param_grid_svm,
    cv=stratified_cv,
    scoring="f1",
    n_jobs=1,
    verbose=1,
)

print("Training SVM pipeline (scaler + model)...")
grid_svm.fit(X_train_smote, y_train_smote)

print(f"Best SVM parameters: {grid_svm.best_params_}")
print(f"Best CV F1 score: {grid_svm.best_score_:.4f}")

Training SVM (may take several minutes)...
Fitting 5 folds for each of 3 candidates, totalling 15 fits


KeyboardInterrupt: 

In [ ]:
# Evaluate SVM on the held-out test set (unscaled — pipeline handles scaling)
y_svm_pred = grid_svm.predict(X_test)
svm_metrics = model_metrics(y_test, y_svm_pred)

print("SVM Test Set Results:")
for metric, value in svm_metrics.items():
    print(f"  {metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_svm_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_svm_pred, target_names=["No Sepsis", "Sepsis"]))

### 4.2 K-Nearest Neighbors (KNN)

In [ ]:
# 4.2 KNN — Elwin Goh Yao Zu
# KNN is slow on 850k rows — tune on a 50k stratified subset (pipeline still includes scaler)
X_knn, _, y_knn, _ = train_test_split(
    X_train_smote,
    y_train_smote,
    train_size=50000,
    stratify=y_train_smote,
    random_state=42,
)

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_jobs=1)),
])

param_grid_knn = {
    "knn__n_neighbors": [5, 11],
    "knn__weights": ["uniform", "distance"],
}

grid_knn = GridSearchCV(
    knn_pipeline,
    param_grid_knn,
    cv=stratified_cv,
    scoring="f1",
    n_jobs=1,
    verbose=1,
)

print(f"Training KNN pipeline on {X_knn.shape[0]} rows...")
grid_knn.fit(X_knn, y_knn)

print(f"Best KNN parameters: {grid_knn.best_params_}")
print(f"Best CV F1 score: {grid_knn.best_score_:.4f}")

Training KNN (may take a while on large data — use Google Colab if needed)...
Fitting 5 folds for each of 8 candidates, totalling 40 fits


KeyboardInterrupt: 

In [ ]:
# Evaluate KNN on the held-out test set (unscaled — pipeline handles scaling)
y_knn_pred = grid_knn.predict(X_test)
knn_metrics = model_metrics(y_test, y_knn_pred)

print("KNN Test Set Results:")
for metric, value in knn_metrics.items():
    print(f"  {metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_knn_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_knn_pred, target_names=["No Sepsis", "Sepsis"]))

### 4.3 Decision Tree

In [ ]:
# 4.3 Decision Tree — Kaizen Soh
dt_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("dt", DecisionTreeClassifier(random_state=42)),
])

param_grid_dt = {
    "dt__max_depth": [10, 15, 20],
    "dt__min_samples_split": [2, 10, 50],
}

grid_dt = GridSearchCV(
    dt_pipeline,
    param_grid_dt,
    cv=stratified_cv,
    scoring="f1",
    n_jobs=1,
    verbose=1,
)

print("Training Decision Tree pipeline...")
grid_dt.fit(X_train_smote, y_train_smote)

print(f"Best Decision Tree parameters: {grid_dt.best_params_}")
print(f"Best CV F1 score: {grid_dt.best_score_:.4f}")

In [ ]:
# Evaluate Decision Tree on the held-out test set (unscaled — pipeline handles scaling)
y_dt_pred = grid_dt.predict(X_test)
dt_metrics = model_metrics(y_test, y_dt_pred)

print("Decision Tree Test Set Results:")
for metric, value in dt_metrics.items():
    print(f"  {metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_dt_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_dt_pred, target_names=["No Sepsis", "Sepsis"]))

### 4.4 Model Comparison

Compare all three models on the same test set. For sepsis, **Recall** and **FNR** are especially important.

In [ ]:
comparison_df = pd.DataFrame(
    {
        "Model": ["SVM", "KNN", "Decision Tree"],
        **{metric: [svm_metrics[metric], knn_metrics[metric], dt_metrics[metric]] for metric in svm_metrics},
    }
)

display(comparison_df.round(4))

best_model = comparison_df.loc[comparison_df["F1"].idxmax(), "Model"]
print(f"\nBest model by F1 score: {best_model}")

### 4.5 Save All Models

Like senior project: each `.pkl` contains **scaler + model** (no separate `scaler.pkl`).
Run this **once at the end** after all models are trained and evaluated.

In [ ]:
# Save models — each file includes scaler + model (Pipeline inside GridSearchCV)
joblib.dump(grid_svm, os.path.join(MODEL_DIR, "svm_model.pkl"))
joblib.dump(grid_knn, os.path.join(MODEL_DIR, "knn_model.pkl"))
joblib.dump(grid_dt, os.path.join(MODEL_DIR, "dt_model.pkl"))

print("Saved to DataTraining/models/:")
print("  - svm_model.pkl  (scaler + SVM)")
print("  - knn_model.pkl  (scaler + KNN)")
print("  - dt_model.pkl   (scaler + Decision Tree)")